# MiniCPM5 vs Qwen3.5 as Jev-style tool routers — zero-shot comparison

**Questions:** before spending time on LoRA,
1. does MiniCPM5-1B beat its same-size rival (Qwen3.5-0.8B) at *deciding which tool to call*?
2. how close does MiniCPM5-2B get to Qwen3.5-4B and Qwen3.5-9B — models 1.6× and 3.6× its size?
3. are their probabilities honest (calibrated)?

**How it works (Jev-style, no text generation):**
1. Each test case = a conversation + the available tools, shown as lettered options **plus a `NONE` option**.
2. One forward pass. We read the next-token probabilities of the option letters (`A`, `B`, `C`…) and renormalise them → a choice **and** a probability.
3. We score accuracy, calibration (ECE, NLL, Brier), selective prediction (accuracy when you only trust confident answers), and latency.

**Models:** `openbmb/MiniCPM5-1B`, `openbmb/MiniCPM5-2B` vs `Qwen/Qwen3.5-0.8B`, `Qwen/Qwen3.5-4B`, `Qwen/Qwen3.5-9B`. (Optional: `MiniCPM5-1B-SFT`, the checkpoint before RL, to test whether RL hurt calibration.)

**Data:** [Berkeley Function Calling Leaderboard](https://github.com/ShishirPatil/gorilla/tree/main/berkeley-function-call-leaderboard) v4, pinned commit:
- `multiple` — pick the right function out of 2–4
- `live_multiple` — real user queries, 2–37 functions
- `irrelevance` / `live_irrelevance` — no tool fits; correct answer is `NONE`

### Run it on Kaggle
1. **Settings → Accelerator → GPU T4 ×2** (needed: Qwen3.5-9B is split across both GPUs). **Settings → Internet → On.**
2. **Save Version → "Save & Run All (Commit)"** so it runs in the background and survives closing the tab. Expect roughly 2–3 hours for all five models (the 9B dominates). Set `QUICK = True` for a ~30-minute smoke test first.
3. Each model's results are saved as soon as it finishes, so a crash or timeout doesn't lose finished models: rerun and they load from cache. Everything lands in `/kaggle/working/` (the version's **Output** tab).

**Already ran it once?** Don't redo the 2–3 hours. If your interactive session is still open, `results_*.json` are already in `/kaggle/working/`. Otherwise, in the right-hand panel use **Add Input → Your Work → (the earlier version) → its output**; the results load automatically and only the analysis plus section 7 runs (≈30 min, mostly the format test in 7b).

On T4 / P100 all models run in float16 so the comparison uses one precision; a model that overflows in float16 is automatically rerun in float32.

In [ ]:
# Transformers >= 5.6 is required by MiniCPM5-1B.
# If imports fail after this cell, use "Restart & Run All" once.
!pip install -q -U "transformers>=5.6" accelerate peft
!pip uninstall -y -q torchao   # Kaggle's torchao 0.10 is too old for current peft
import sys
for _m in [m for m in sys.modules if m.split(".")[0] in ("transformers", "peft", "accelerate", "torchao")]:
    del sys.modules[_m]

## 1. Config

In [ ]:
# Run order = this order; the biggest model goes last so smaller results are saved first.
# "base::adapter" loads a LoRA adapter on top of the base model; "auto" finds the adapter in /kaggle/input
# (attach the step 3 notebook's output, or a dataset made from router_adapter.zip). A Hub repo id also works.
ROUTER = "MiniCPM5-2B-Router"
MODELS = {
    "MiniCPM5-2B": "openbmb/MiniCPM5-2B",            # the untrained base, for a paired before/after comparison
    ROUTER:        "openbmb/MiniCPM5-2B::auto",      # base + your LoRA adapter from step 3
}
COMPARE_ALL = False   # True: also re-run MiniCPM5-1B and Qwen3.5 0.8B/4B/9B (~3 h); False: use the first run's numbers
if COMPARE_ALL:
    MODELS = {"MiniCPM5-1B": "openbmb/MiniCPM5-1B", "MiniCPM5-2B": "openbmb/MiniCPM5-2B", ROUTER: MODELS[ROUTER],
              "Qwen3.5-0.8B": "Qwen/Qwen3.5-0.8B", "Qwen3.5-4B": "Qwen/Qwen3.5-4B", "Qwen3.5-9B": "Qwen/Qwen3.5-9B"}
# Nominal sizes in billions of parameters (for the size-vs-accuracy chart).
PARAMS_B = {"MiniCPM5-1B": 1.1, "MiniCPM5-1B-SFT": 1.1, "MiniCPM5-2B": 2.5, ROUTER: 2.5,
            "Qwen3.5-0.8B": 0.8, "Qwen3.5-4B": 4.0, "Qwen3.5-9B": 9.0}
FAMILY = lambda name: "MiniCPM5 + LoRA" if "Router" in name else name.split("-")[0]   # "MiniCPM5" / "Qwen3.5"

# Head-to-heads to test with a paired bootstrap: (challenger, reference)
PAIRS = [(ROUTER, "MiniCPM5-2B"),            # did the LoRA help?
         (ROUTER, "Qwen3.5-4B"), (ROUTER, "Qwen3.5-9B"),
         ("MiniCPM5-1B", "Qwen3.5-0.8B"), ("MiniCPM5-2B", "Qwen3.5-4B"), ("MiniCPM5-2B", "Qwen3.5-9B")]

QUICK = False   # True = small sample per category for a fast smoke test

# Examples per BFCL category (None = use all). Totals: 200 / 1053 / 240 / 884.
SAMPLES_PER_CATEGORY = {
    "multiple": 200,
    "live_multiple": 400,
    "irrelevance": 240,
    "live_irrelevance": 300,
}
if QUICK:
    SAMPLES_PER_CATEGORY = {k: 40 for k in SAMPLES_PER_CATEGORY}

MAX_TOOLS = 25            # options are letters A..Z, one is reserved for NONE
MAX_PROMPT_TOKENS = 4096  # longer prompts are skipped (for every model, to keep it fair)
DESC_CHARS = 300          # tool descriptions are trimmed to this length
N_PERMUTATIONS = 1        # >1 averages over rotated tool orders to cancel position bias (costs N x time)
SEED = 42
OUT_DIR = "/kaggle/working"
DELETE_WEIGHTS_AFTER = True   # free disk: the five checkpoints together are ~40 GB

BFCL_COMMIT = "6ea57973c7a6097fd7c5915698c54c17c5b1b6c8"
BFCL_BASE = ("https://raw.githubusercontent.com/ShishirPatil/gorilla/"
             f"{BFCL_COMMIT}/berkeley-function-call-leaderboard/bfcl_eval/data/")

In [ ]:
import os, gc, glob, json, math, time, random, shutil, urllib.request
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Precision: bf16 where the GPU supports it natively (Ampere+). T4 / P100 have no native bf16,
# so every model runs in fp16 (one precision for a fair comparison; the 9B only fits in 16-bit),
# and a model whose logits overflow in fp16 is rerun in fp32.
if DEVICE == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8:
    DTYPES = [torch.bfloat16]
elif DEVICE == "cuda":
    DTYPES = [torch.float16, torch.float32]
else:
    DTYPES = [torch.float32]

N_GPUS = torch.cuda.device_count() if DEVICE == "cuda" else 0
print("torch", torch.__version__, "| device", DEVICE,
      f"{N_GPUS} x {torch.cuda.get_device_name(0)}" if N_GPUS else "", "| dtypes", DTYPES)
try:
    import transformers
    print("transformers", transformers.__version__)
except ImportError:
    pass

## 2. Load BFCL and turn it into decision questions

In [ ]:
def fetch_jsonl(rel_path):
    local = os.path.join(OUT_DIR, "bfcl", rel_path.replace("/", "__"))
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.exists(local):
        urllib.request.urlretrieve(BFCL_BASE + rel_path, local)
    with open(local, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def render_tool(fn):
    params = fn.get("parameters") or {}
    props = params.get("properties") or {}
    required = set(params.get("required") or [])
    args = []
    for name, spec in props.items():
        typ = spec.get("type", "any") if isinstance(spec, dict) else "any"
        args.append(f"{name}{'' if name in required else '?'}: {typ}")
    desc = " ".join(str(fn.get("description", "")).split())
    if len(desc) > DESC_CHARS:
        desc = desc[:DESC_CHARS].rstrip() + "..."
    return f"{fn['name']}({', '.join(args)}) - {desc}"


def render_conversation(messages):
    return "\n".join(f"{m['role'].upper()}: {m['content']}" for m in messages)


rng = random.Random(SEED)
examples = []
for cat, n in SAMPLES_PER_CATEGORY.items():
    data = fetch_jsonl(f"BFCL_v4_{cat}.json")
    gold_names = {}
    if cat in ("multiple", "live_multiple"):
        for a in fetch_jsonl(f"possible_answer/BFCL_v4_{cat}.json"):
            gold_names[a["id"]] = next(iter(a["ground_truth"][0]))
    data = [x for x in data if x["function"]]            # a few live_irrelevance rows list no tools
    if n is not None and n < len(data):
        data = rng.sample(data, n)

    for x in data:
        funcs = list(x["function"])
        gold = gold_names.get(x["id"])                    # None -> no tool fits -> NONE
        truncated = len(funcs) > MAX_TOOLS
        if truncated:
            keep = [f for f in funcs if f["name"] == gold]
            others = [f for f in funcs if f["name"] != gold]
            funcs = keep + rng.sample(others, MAX_TOOLS - len(keep))
        rng.shuffle(funcs)                                # same order for every model
        names = [f["name"] for f in funcs]
        examples.append({
            "id": x["id"],
            "category": cat,
            "relevant": gold is not None,
            "conversation": render_conversation(x["question"][0]),
            "tool_lines": [render_tool(f) for f in funcs],
            "options": names + ["NONE"],
            "gold_idx": names.index(gold) if gold is not None else len(names),
            "truncated": truncated,
        })

ex_df = pd.DataFrame(examples)
print(ex_df.groupby("category").agg(n=("id", "size"),
                                    mean_options=("options", lambda s: s.map(len).mean()),
                                    truncated=("truncated", "sum")))

The prompt is identical for every model. Tool order is shuffled once (seeded) and `NONE` is always the last letter.

In [ ]:
LETTERS = [chr(ord("A") + i) for i in range(26)]
SYSTEM = ("You are a tool-routing decision model. Read the conversation and the lettered options, "
          "then choose the ONE option that should happen next: the single tool to call, or NONE if "
          "no listed tool can fulfil the request. Reply with the option letter only.")
PREFILL = "Answer:"


def user_message(ex, order):
    # order: indices into ex['tool_lines'] (lets us rotate tools for position-bias averaging)
    lines = [f"({LETTERS[i]}) {ex['tool_lines'][j]}" for i, j in enumerate(order)]
    lines.append(f"({LETTERS[len(order)]}) NONE - none of the tools above fits this request")
    return ("Conversation:\n" + ex["conversation"] + "\n\nOptions:\n" + "\n".join(lines)
            + "\n\nWhich option should happen next? Reply with the letter only.")


def build_prompt(tok, ex, order, prefill=None):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user_message(ex, order)}]
    kwargs = dict(tokenize=False, add_generation_prompt=True, enable_thinking=False)
    try:
        text = tok.apply_chat_template(msgs, **kwargs)
    except Exception:  # template without a system role
        merged = [{"role": "user", "content": SYSTEM + "\n\n" + msgs[1]["content"]}]
        text = tok.apply_chat_template(merged, **kwargs)
    return text + (PREFILL if prefill is None else prefill)


print(user_message(examples[0], list(range(len(examples[0]["tool_lines"])))))

## 3. Score each model with one forward pass per question

In [ ]:
def letter_token_ids(tok, prompt):
    # Find the token id each letter gets right after the prefill ("Answer: A" or "Answer:A").
    base = tok(prompt, add_special_tokens=False)["input_ids"]
    for joiner in (" ", ""):
        ids = []
        for L in LETTERS:
            full = tok(prompt + joiner + L, add_special_tokens=False)["input_ids"]
            if len(full) != len(base) + 1 or full[:len(base)] != base:
                break
            ids.append(full[-1])
        if len(ids) == len(LETTERS) and len(set(ids)) == len(LETTERS):
            return joiner, ids
    raise ValueError("Option letters are not single, distinct tokens after the prefill")


def find_adapter(spec):
    if spec != "auto":
        return spec                                   # a local folder or a Hub repo id
    hits = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)) + \
           sorted(glob.glob(os.path.join(OUT_DIR, "**/adapter_config.json"), recursive=True))
    best = [h for h in hits if os.sep + "adapter" + os.sep in h and "adapter_last" not in h] or hits
    if not best:
        raise FileNotFoundError("No LoRA adapter found: attach the step 3 output (Add Input -> Your Work)")
    return os.path.dirname(best[0])


def load_model(repo, dtype):
    if "::" in repo:                                  # base model + LoRA adapter, merged for fast scoring
        base, spec = repo.split("::", 1)
        tok, model = load_model(base, dtype)
        from peft import PeftModel
        path = find_adapter(spec)
        print("loading adapter from", path)
        model = PeftModel.from_pretrained(model, path).merge_and_unload().eval()
        return tok, model
    import transformers
    tok = transformers.AutoTokenizer.from_pretrained(repo)
    errors = []
    # device_map="auto" fills GPU 0 first and only splits big models (the 9B) across both T4s.
    kw = dict(dtype=dtype, device_map="auto") if DEVICE == "cuda" else dict(dtype=dtype)
    # Qwen3.5 checkpoints are vision-language models; a text-only forward works either way.
    for cls_name in ("AutoModelForCausalLM", "AutoModelForImageTextToText", "AutoModelForMultimodalLM"):
        cls = getattr(transformers, cls_name, None)
        if cls is None:
            continue
        try:
            model = cls.from_pretrained(repo, **kw).eval()
            return tok, model
        except Exception as e:
            errors.append(f"{cls_name}: {type(e).__name__}: {e}")
    raise RuntimeError("Could not load " + repo + "\n" + "\n".join(errors))


def input_device(model):
    try:
        return model.get_input_embeddings().weight.device
    except Exception:
        return torch.device(DEVICE)


@torch.inference_mode()
def next_token_logprobs(model, input_ids):
    try:
        out = model(input_ids=input_ids, logits_to_keep=1)
    except TypeError:
        out = model(input_ids=input_ids)
    return torch.log_softmax(out.logits[0, -1].float(), dim=-1)


def delete_cached_weights(repo):
    repo = repo.split("::")[0]
    try:
        from huggingface_hub.constants import HF_HUB_CACHE
    except ImportError:
        return
    shutil.rmtree(os.path.join(HF_HUB_CACHE, "models--" + repo.replace("/", "--")), ignore_errors=True)


def free_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def score_all(name, tok, model, dtype):
    dev = input_device(model)
    joiner, letter_ids = letter_token_ids(tok, build_prompt(tok, examples[0], [0]))
    print(f"{name} [{dtype}]: letters follow the prefill with joiner {joiner!r}")

    # Warm-up so the first timed example isn't paying for CUDA init.
    warm = tok(build_prompt(tok, examples[0], [0]), add_special_tokens=False,
               return_tensors="pt")["input_ids"].to(dev)
    for _ in range(3):
        next_token_logprobs(model, warm)

    records = []
    for ex in tqdm(examples, desc=name):
        n_tools = len(ex["tool_lines"])
        n_opts = n_tools + 1
        probs = np.zeros(n_opts)
        latency, n_tokens, mass, skipped = 0.0, 0, 0.0, False
        for p in range(N_PERMUTATIONS):
            shift = (p * n_tools) // N_PERMUTATIONS
            order = list(range(n_tools))[shift:] + list(range(n_tools))[:shift]
            ids = tok(build_prompt(tok, ex, order), add_special_tokens=False,
                      return_tensors="pt")["input_ids"]
            if ids.shape[1] > MAX_PROMPT_TOKENS:
                skipped = True
                break
            ids = ids.to(dev)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            logp = next_token_logprobs(model, ids)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            latency += time.perf_counter() - t0
            n_tokens = ids.shape[1]
            lp = logp[letter_ids[:n_opts]].double()
            if not torch.isfinite(lp).all():
                raise FloatingPointError(f"non-finite logits on {ex['id']}")
            mass += lp.exp().sum().item() / N_PERMUTATIONS       # how much probability sits on valid letters
            p_perm = torch.softmax(lp, 0).cpu().numpy()
            # map positions back to tool identity (NONE stays last)
            p_tool = np.empty(n_opts)
            p_tool[order] = p_perm[:n_tools]
            p_tool[n_tools] = p_perm[n_tools]
            probs += p_tool / N_PERMUTATIONS
        if skipped:
            records.append({"model": name, "id": ex["id"], "skipped": True})
            continue
        pred = int(np.argmax(probs))
        records.append({
            "model": name, "id": ex["id"], "category": ex["category"], "relevant": ex["relevant"],
            "skipped": False, "n_options": n_opts, "gold_idx": ex["gold_idx"], "pred_idx": pred,
            "correct": pred == ex["gold_idx"], "confidence": float(probs[pred]),
            "p_gold": float(probs[ex["gold_idx"]]), "probs": probs.tolist(),
            "letter_mass": mass, "latency_ms": 1000 * latency / N_PERMUTATIONS,
            "prompt_tokens": n_tokens, "dtype": str(dtype),
        })
    return records


def run_model(name, repo):
    for i, dtype in enumerate(DTYPES):
        tok, model = load_model(repo, dtype)
        try:
            if i == 0:   # show one real prompt so you can eyeball the chat template
                print(build_prompt(tok, examples[0], list(range(len(examples[0]["tool_lines"])))))
            records = score_all(name, tok, model, dtype)
            break
        except FloatingPointError as e:
            if i == len(DTYPES) - 1:
                raise
            print(f"{name}: {e} in {dtype}; rerunning in {DTYPES[i + 1]}")
        finally:
            del model
            free_memory()
    with open(os.path.join(OUT_DIR, f"results_{name}.json"), "w") as f:
        json.dump(records, f)
    return records

In [ ]:
import glob


def find_cached(name):
    # This session's output first, then any attached input (e.g. a previous version's output).
    candidates = [os.path.join(OUT_DIR, f"results_{name}.json")]
    candidates += sorted(glob.glob(f"/kaggle/input/**/results_{name}.json", recursive=True))
    return next((c for c in candidates if os.path.exists(c)), None)


all_records = []
for name, repo in MODELS.items():
    path = find_cached(name)
    if path:                                       # resume / reuse a previous run
        print(f"{name}: loading cached results from {path}")
        with open(path) as f:
            all_records += json.load(f)
        continue
    try:
        all_records += run_model(name, repo)
    except Exception as e:                         # e.g. out of memory: keep going with the others
        print(f"!! {name} failed and is left out of the comparison: {type(e).__name__}: {e}")
        free_memory()
    if DELETE_WEIGHTS_AFTER:
        delete_cached_weights(repo)

## 4. Results

In [ ]:
from scipy.optimize import minimize_scalar

raw = pd.DataFrame(all_records)
model_names = [m for m in MODELS if m in set(raw["model"])]
# Fair comparison: keep only questions every model scored.
skipped_ids = set(raw.loc[raw["skipped"] == True, "id"])
res = raw[(raw["skipped"] == False) & ~raw["id"].isin(skipped_ids)].copy()
shared = set.intersection(*[set(res.loc[res["model"] == m, "id"]) for m in model_names])
res = res[res["id"].isin(shared)].copy()
res = res.astype({"relevant": bool, "correct": bool, "gold_idx": int, "pred_idx": int, "n_options": int})
print(f"{len(shared)} questions scored by all {len(model_names)} models "
      f"({len(skipped_ids)} skipped as too long)")

# One calibration/test split shared by every model (stratified by category).
split_rng = np.random.default_rng(SEED)
calib_ids = set()
for cat, ids in ex_df[ex_df["id"].isin(shared)].groupby("category")["id"]:
    ids = sorted(ids)
    calib_ids |= set(split_rng.choice(ids, size=len(ids) // 2, replace=False))
res["split"] = np.where(res["id"].isin(calib_ids), "calib", "test")


def ece(conf, correct, n_bins=15):
    conf, correct = np.asarray(conf), np.asarray(correct, dtype=float)
    bins = np.minimum((conf * n_bins).astype(int), n_bins - 1)
    total = 0.0
    for b in range(n_bins):
        m = bins == b
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return total


def apply_temperature(probs_list, T):
    out = []
    for p in probs_list:
        z = np.log(np.clip(np.asarray(p), 1e-12, 1)) / T
        z = np.exp(z - z.max())
        out.append(z / z.sum())
    return out


def nll(probs_list, gold):
    return float(np.mean([-math.log(max(p[g], 1e-12)) for p, g in zip(probs_list, gold)]))


def brier(probs_list, gold):
    return float(np.mean([((np.asarray(p) - np.eye(len(p))[g]) ** 2).sum()
                          for p, g in zip(probs_list, gold)]))


def risk_coverage(conf, correct):
    order = np.argsort(-np.asarray(conf), kind="stable")
    c = np.asarray(correct, dtype=float)[order]
    acc_at = np.cumsum(c) / np.arange(1, len(c) + 1)
    coverage = np.arange(1, len(c) + 1) / len(c)
    return coverage, acc_at


def fit_temperature(probs_list, gold):
    r = minimize_scalar(lambda T: nll(apply_temperature(probs_list, T), gold),
                        bounds=(0.05, 20.0), method="bounded")
    return float(r.x)


rows, curves, temps = [], {}, {}
for m in model_names:
    d = res[res["model"] == m]
    cal, test = d[d["split"] == "calib"], d[d["split"] == "test"]
    T = fit_temperature(list(cal["probs"]), list(cal["gold_idx"]))
    temps[m] = T
    tp = apply_temperature(list(test["probs"]), T)
    t_conf = [p.max() for p in tp]
    cov, acc_at = risk_coverage(d["confidence"], d["correct"])
    curves[m] = (cov, acc_at)
    k50 = max(1, len(d) // 2)
    rows.append({
        "model": m,
        "params (B)": PARAMS_B.get(m, float("nan")),
        "accuracy": d["correct"].mean(),
        "tool pick acc (relevant)": d.loc[d["relevant"], "correct"].mean(),
        "NONE recall (irrelevant)": d.loc[~d["relevant"], "correct"].mean(),
        "false NONE rate (relevant)": (d.loc[d["relevant"], "pred_idx"]
                                       == d.loc[d["relevant"], "n_options"] - 1).mean(),
        "mean confidence": d["confidence"].mean(),
        "ECE": ece(d["confidence"], d["correct"]),
        "NLL": nll(list(d["probs"]), list(d["gold_idx"])),
        "Brier": brier(list(d["probs"]), list(d["gold_idx"])),
        "fitted T": T,
        "test ECE raw": ece(test["confidence"], test["correct"]),
        "test ECE after T": ece(t_conf, test["correct"]),
        "acc @50% coverage": acc_at[k50 - 1],
        "AURC (lower=better)": float(np.mean(1 - acc_at)),
        "letter mass": d["letter_mass"].mean(),
        "median latency ms": d["latency_ms"].median(),
        "median prompt tokens": d["prompt_tokens"].median(),
        "dtype": d["dtype"].iloc[0].replace("torch.", "") if "dtype" in d else "",
    })

summary = pd.DataFrame(rows).set_index("model")
summary.to_csv(os.path.join(OUT_DIR, "summary.csv"))
pd.options.display.float_format = "{:.3f}".format
summary.T

**How to read it**
- **accuracy / tool pick acc** — is it choosing the right tool? This is the hard part to fix later.
- **ECE** — gap between stated confidence and real accuracy (0 = perfectly honest). `test ECE after T` shows how much one temperature number fixes it.
- **fitted T** — >1 means the model was over-confident, <1 under-confident.
- **acc @50% coverage / AURC** — if you only trust its most confident half of answers, how good is it? This is what a router with "escalate to a big model when unsure" actually uses.
- **letter mass** — share of probability on valid option letters. Low values mean the model wanted to say something else (format problem, often fixed by LoRA).

In [ ]:
by_cat = (res.groupby(["category", "model"])["correct"].mean().unstack("model")[model_names]
          .reindex(list(SAMPLES_PER_CATEGORY)))
by_cat["n questions"] = res[res["model"] == model_names[0]].groupby("category").size()
by_cat

In [ ]:
# Paired bootstrap on the same questions: is the challenger really better, or is it noise?
boot_rng = np.random.default_rng(SEED)
correct_by = {m: res[res["model"] == m].set_index("id")["correct"].astype(float) for m in model_names}


def boot_ci(diff, n=5000):
    means = [diff[boot_rng.integers(0, len(diff), len(diff))].mean() for _ in range(n)]
    return np.percentile(means, [2.5, 97.5])


verdicts = []
for challenger, reference in PAIRS:
    if challenger not in model_names or reference not in model_names:
        print(f"skip {challenger} vs {reference}: not both scored")
        continue
    a = correct_by[challenger]
    b = correct_by[reference].reindex(a.index)
    diff = (a - b).to_numpy()
    lo, hi = boot_ci(diff)
    print(f"{challenger} - {reference} accuracy: {diff.mean():+.3f}  (95% CI {lo:+.3f} to {hi:+.3f})")
    for cat in SAMPLES_PER_CATEGORY:
        ids = res[(res["model"] == challenger) & (res["category"] == cat)]["id"]
        dc = (a.loc[ids] - b.loc[ids]).to_numpy()
        l2, h2 = boot_ci(dc)
        print(f"  {cat:17s} {dc.mean():+.3f}  (95% CI {l2:+.3f} to {h2:+.3f}, n={len(dc)})")
    verdict = (f"{challenger} is better" if lo > 0 else
               f"{reference} is better" if hi < 0 else
               "no significant difference")
    verdicts.append({"challenger": challenger, "reference": reference,
                     "size ratio": PARAMS_B[reference] / PARAMS_B[challenger],
                     "accuracy diff": diff.mean(), "95% CI low": lo, "95% CI high": hi,
                     "verdict": verdict})
    print()

verdict_df = pd.DataFrame(verdicts)
verdict_df.to_csv(os.path.join(OUT_DIR, "head_to_head.csv"), index=False)
verdict_df

## 5. Charts

In [ ]:
import matplotlib.pyplot as plt

# Fixed categorical order (blue, orange, aqua, yellow, magenta, green); colour follows the model.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
color = {m: SERIES[i % len(SERIES)] for i, m in enumerate(MODELS)}
FAMILY_COLOR = {"MiniCPM5": SERIES[0], "Qwen3.5": SERIES[1], "MiniCPM5 + LoRA": SERIES[2]}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": GRID,
    "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True, "font.size": 10,
})


def reliability(conf, correct, n_bins=10):
    conf, correct = np.asarray(conf), np.asarray(correct, dtype=float)
    bins = np.minimum((conf * n_bins).astype(int), n_bins - 1)
    xs, ys, ns = [], [], []
    for b in range(n_bins):
        m = bins == b
        if m.sum() >= 5:                          # skip near-empty bins
            xs.append(conf[m].mean()); ys.append(correct[m].mean()); ns.append(m.sum())
    return xs, ys, ns


# 1) Reliability: raw vs temperature-scaled, test split
ncols = min(3, len(model_names))
nrows = math.ceil(len(model_names) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 4 * nrows), sharey=True, squeeze=False)
axes = axes.ravel()
for ax in axes[len(model_names):]:
    ax.set_visible(False)
for ax, m in zip(axes, model_names):
    test = res[(res["model"] == m) & (res["split"] == "test")]
    ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls=":", label="perfect calibration")
    x, y, _ = reliability(test["confidence"], test["correct"])
    ax.plot(x, y, color=color[m], lw=2, marker="o", ms=6, label=f"raw (ECE {ece(test['confidence'], test['correct']):.3f})")
    tc = [p.max() for p in apply_temperature(list(test["probs"]), temps[m])]
    x, y, _ = reliability(tc, test["correct"])
    ax.plot(x, y, color=color[m], lw=2, ls="--", marker="s", ms=6, mfc="#fcfcfb",
            label=f"T={temps[m]:.2f} (ECE {ece(tc, test['correct']):.3f})")
    ax.set_title(m, loc="left", fontsize=11)
    ax.set_xlabel("stated confidence"); ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
    ax.legend(frameon=False, fontsize=8, loc="upper left")
for r in range(nrows):
    axes[r * ncols].set_ylabel("actual accuracy")
fig.suptitle("Calibration on held-out half (closer to the diagonal = more honest)", x=0.01, ha="left")
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "reliability.png"), dpi=150); plt.show()

# 2) Selective prediction: accuracy when only the most confident answers are kept
fig, ax = plt.subplots(figsize=(8, 4.4))
for m in model_names:
    cov, acc_at = curves[m]
    ax.plot(cov * 100, acc_at * 100, color=color[m], lw=2, label=m)
# end labels, nudged apart so they don't overlap
ends = sorted(((curves[m][1][-1] * 100, m) for m in model_names))
lo_y, hi_y = ax.get_ylim()
gap = (hi_y - lo_y) * 0.05
placed = []
for y, m in ends:
    y_lab = max(y, placed[-1] + gap) if placed else y
    placed.append(y_lab)
    ax.annotate(f"{m}  {y:.1f}%", (100, y), xytext=(108, y_lab), textcoords="data",
                va="center", color=INK, fontsize=9, annotation_clip=False,
                arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
ax.set_xlabel("coverage: % of questions answered (most confident first)")
ax.set_ylabel("accuracy on answered questions (%)")
ax.set_xlim(0, 100); ax.legend(frameon=False, loc="lower left")
ax.set_title("Selective prediction: answer when confident, escalate the rest", loc="left", fontsize=11)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "risk_coverage.png"), dpi=150); plt.show()

# 3) Accuracy by category
cats = list(SAMPLES_PER_CATEGORY)
acc = by_cat.drop(columns="n questions")
w = 0.8 / len(model_names)
fig, ax = plt.subplots(figsize=(10, 4.4))
for i, m in enumerate(model_names):
    xs = np.arange(len(cats)) + (i - (len(model_names) - 1) / 2) * w
    ax.bar(xs, acc[m] * 100, width=w - 0.02, color=color[m], label=m, edgecolor="#fcfcfb", linewidth=2)
ax.set_xticks(np.arange(len(cats)), cats)
ax.grid(axis="x", visible=False)
ax.set_ylabel("accuracy (%)"); ax.set_ylim(0, 100)
ax.legend(frameon=False, ncol=min(5, len(model_names)), loc="upper left", bbox_to_anchor=(0, 1.12))
ax.set_title("Accuracy by BFCL category", loc="left", fontsize=11, pad=28)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "accuracy_by_category.png"), dpi=150); plt.show()

# 4) Size vs accuracy: does MiniCPM5 punch above its weight?
fig, ax = plt.subplots(figsize=(7.5, 4.4))
for fam, fc in FAMILY_COLOR.items():
    ms = sorted([m for m in model_names if FAMILY(m) == fam and m in PARAMS_B and "SFT" not in m],
                key=PARAMS_B.get)
    if not ms:
        continue
    xs = [PARAMS_B[m] for m in ms]
    ys = [summary.loc[m, "accuracy"] * 100 for m in ms]
    ax.plot(xs, ys, color=fc, lw=2, marker="o", ms=8, mec="#fcfcfb", mew=2, label=fam)
    for m, x, y in zip(ms, xs, ys):
        ax.annotate(f"{m.split('-', 1)[1]}  {y:.1f}%", (x, y), xytext=(0, 10),
                    textcoords="offset points", ha="center", fontsize=9, color=INK)
ax.set_xscale("log")
ax.set_xticks([0.8, 1, 2, 4, 9], ["0.8B", "1B", "2B", "4B", "9B"])
ax.minorticks_off()
ax.set_xlabel("parameters (log scale)"); ax.set_ylabel("overall accuracy (%)")
ax.legend(frameon=False, loc="lower right")
ax.set_title("Accuracy vs model size", loc="left", fontsize=11)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "size_vs_accuracy.png"), dpi=150); plt.show()

## 6. Where it goes wrong

In [ ]:
# The most confident mistakes of a MiniCPM model: the cases a router would wrongly trust.
m = next((x for x in ("MiniCPM5-2B", "MiniCPM5-1B") if x in model_names), model_names[0])
print("Model:", m, "\n")
ex_by_id = {e["id"]: e for e in examples}
worst = res[(res["model"] == m) & (~res["correct"])].sort_values("confidence", ascending=False).head(8)
for _, r in worst.iterrows():
    e = ex_by_id[r["id"]]
    print(f"[{e['category']}] {e['id']}  confidence {r['confidence']:.2f}")
    print("  user :", e["conversation"][:200].replace("\n", " "))
    print("  gold :", e["options"][r["gold_idx"]])
    print("  pred :", e["options"][r["pred_idx"]], "\n")

## 7. Diagnosis: why does MiniCPM5 miss NONE, and is the answer format hurting it?

The first run showed MiniCPM5 picks tools as well as much bigger Qwen models but answers NONE far less often, and that it puts little probability on the option letters ("letter mass"). Two tests separate *fixable bias* from *real weakness*:

- **7a. NONE-bias test (instant, no GPU):** add one number to the NONE option's score, tuned on the calibration half, and re-measure on the held-out half. If the gap closes, MiniCPM just has a bias toward calling tools that a tiny correction (or LoRA) removes. If it doesn't, it genuinely can't tell when no tool fits.
- **7b. Answer-format test (~30 min):** on 100 questions, see which tokens each model wants to output after the prompt, and re-score with alternative answer formats. If accuracy jumps with a different format, the main results undersold that model.

In [ ]:
def apply_none_bias(probs_list, b):
    out = []
    for p in probs_list:
        z = np.log(np.clip(np.asarray(p, dtype=float), 1e-12, 1))
        z[-1] += b                                    # NONE is always the last option
        z = np.exp(z - z.max())
        out.append(z / z.sum())
    return out


def routing_stats(probs_list, frame):
    pred = np.array([int(np.argmax(p)) for p in probs_list])
    gold = frame["gold_idx"].to_numpy()
    rel = frame["relevant"].to_numpy()
    none_idx = frame["n_options"].to_numpy() - 1
    ok = pred == gold
    return {"accuracy": ok.mean(), "tool pick acc": ok[rel].mean(), "NONE recall": ok[~rel].mean(),
            "false NONE rate": (pred[rel] == none_idx[rel]).mean(), "correct": ok}


GRID = np.arange(-4, 12.01, 0.25)
bias_rows, bias_correct = [], {}
for m in model_names:
    d = res[res["model"] == m]
    cal, test = d[d["split"] == "calib"], d[d["split"] == "test"]
    cal_acc = [routing_stats(apply_none_bias(list(cal["probs"]), b), cal)["accuracy"] for b in GRID]
    best = max(cal_acc)
    b = min((g for g, a in zip(GRID, cal_acc) if a == best), key=abs)   # smallest shift that reaches the best
    before = routing_stats(list(test["probs"]), test)
    after_p = apply_none_bias(list(test["probs"]), b)
    after = routing_stats(after_p, test)
    bias_correct[m] = pd.Series(after["correct"].astype(float), index=test["id"].to_numpy())
    bias_rows.append({"model": m, "NONE bias (log-odds)": b,
                      "test acc before": before["accuracy"], "test acc after": after["accuracy"],
                      "tool pick before": before["tool pick acc"], "tool pick after": after["tool pick acc"],
                      "NONE recall before": before["NONE recall"], "NONE recall after": after["NONE recall"],
                      "false NONE after": after["false NONE rate"],
                      "test ECE after bias": ece([p.max() for p in after_p], after["correct"])})

bias_df = pd.DataFrame(bias_rows).set_index("model")
bias_df.to_csv(os.path.join(OUT_DIR, "none_bias_fix.csv"))
display(bias_df.T)

print("Head-to-head on the held-out half, after each model gets its own NONE correction:")
for challenger, reference in PAIRS:
    if challenger in bias_correct and reference in bias_correct:
        a = bias_correct[challenger]
        diff = (a - bias_correct[reference].reindex(a.index)).to_numpy()
        lo, hi = boot_ci(diff)
        verdict = (f"{challenger} better" if lo > 0 else f"{reference} better" if hi < 0 else "no significant difference")
        print(f"  {challenger:12s} vs {reference:12s} {diff.mean():+.3f}  (95% CI {lo:+.3f} to {hi:+.3f})  -> {verdict}")

print("\nHow much of the NONE gap is just bias?")
for m in model_names:
    r = bias_df.loc[m]
    gained = r["NONE recall after"] - r["NONE recall before"]
    lost = r["tool pick before"] - r["tool pick after"]
    print(f"  {m:12s} NONE recall {r['NONE recall before']:.1%} -> {r['NONE recall after']:.1%} "
          f"({gained:+.1%}), tool picking {r['tool pick before']:.1%} -> {r['tool pick after']:.1%} ({-lost:+.1%})")

**Reading 7a:** a large positive bias that lifts NONE recall while tool picking barely drops means the model *knows* when nothing fits but is tilted toward calling tools: LoRA will fix this cheaply. If NONE recall only rises by trading away tool picking one-for-one, the model can't separate the cases and LoRA needs lots of hard negative examples.

In [ ]:
# 7b. Where does the probability go, and does another answer format work better? (~30 min on 2x T4)
RUN_FORMAT_DIAGNOSTIC = False             # done in the first run; True re-runs it (~30 min)
DIAG_PER_CATEGORY = 25                      # 4 categories -> 100 questions per model
DIAG_PREFILLS = {
    "Answer: X":   "Answer:",               # the format used in the main run (letter after a space)
    "Answer: (X":  "Answer: (",             # the options are written "(A) ...", so the model may want "(B)"
    "X (bare)":    "",                      # the letter as the very first word of the reply
}


def letter_ids_for(tok, prompt):
    try:
        return letter_token_ids(tok, prompt)[1]
    except ValueError:
        return None


def diagnose_model(name, repo, sample):
    for i, dtype in enumerate(DTYPES):           # same fp16 -> fp32 fallback as the main run
        try:
            return _diagnose(name, repo, sample, dtype)
        except FloatingPointError:
            if i == len(DTYPES) - 1:
                raise
            print(f"{name}: non-finite logits in {dtype}; retrying in {DTYPES[i + 1]}")


@torch.inference_mode()
def _diagnose(name, repo, sample, dtype):
    tok, model = load_model(repo, dtype)
    dev = input_device(model)
    rows, top = [], {}
    try:
        for label, prefill in DIAG_PREFILLS.items():
            first = build_prompt(tok, sample[0], [0], prefill=prefill)
            ids_letters = letter_ids_for(tok, first)
            if ids_letters is None:
                rows.append({"model": name, "format": label, "note": "letters not single tokens"})
                continue
            correct, mass = [], []
            for ex in sample:
                n_opts = len(ex["tool_lines"]) + 1
                prompt = build_prompt(tok, ex, list(range(n_opts - 1)), prefill=prefill)
                ids = tok(prompt, add_special_tokens=False, return_tensors="pt")["input_ids"].to(dev)
                logp = next_token_logprobs(model, ids)
                lp = logp[ids_letters[:n_opts]].double()
                if not torch.isfinite(lp).all():
                    raise FloatingPointError("non-finite logits")
                mass.append(lp.exp().sum().item())
                correct.append(int(torch.argmax(lp).item()) == ex["gold_idx"])
                if label == "Answer: X":            # what does it want to say instead?
                    probs = logp.exp().float().cpu().numpy()
                    for t in np.argsort(-probs)[:8]:
                        key = repr(tok.decode([int(t)]))
                        top[key] = top.get(key, 0.0) + float(probs[t]) / len(sample)
            rows.append({"model": name, "format": label, "letter mass": np.mean(mass),
                         "accuracy": np.mean(correct)})
    finally:
        del model
        free_memory()
    return rows, sorted(top.items(), key=lambda kv: -kv[1])[:8]


diag_rows, diag_top = [], {}
if RUN_FORMAT_DIAGNOSTIC:
    drng = random.Random(SEED + 1)
    sample = []
    for cat in SAMPLES_PER_CATEGORY:
        pool = [e for e in examples if e["category"] == cat and e["id"] in shared]
        sample += drng.sample(pool, min(DIAG_PER_CATEGORY, len(pool)))
    for name in model_names:
        try:
            r, t = diagnose_model(name, MODELS[name], sample)
            diag_rows += r
            diag_top[name] = t
        except Exception as e:
            print(f"!! {name} diagnostic failed: {type(e).__name__}: {e}")
            free_memory()
        if DELETE_WEIGHTS_AFTER:
            delete_cached_weights(MODELS[name])

    print("Most likely next tokens after the main-run prompt (avg probability over the sample):")
    for name, t in diag_top.items():
        print(f"  {name:12s} " + "  ".join(f"{k} {v:.2f}" for k, v in t))

    diag_df = pd.DataFrame(diag_rows)
    diag_df.to_csv(os.path.join(OUT_DIR, "format_diagnostic.csv"), index=False)
    display(diag_df.pivot(index="format", columns="model", values=["letter mass", "accuracy"])
            .reindex(list(DIAG_PREFILLS)))

In [ ]:
# Plain-English conclusions from 7a and 7b
print("CONCLUSIONS\n")
qwen = [x for x in model_names if x.startswith("Qwen")]
best_qwen_none = bias_df.loc[qwen, "NONE recall before"].max() if qwen else float("nan")
for m in [x for x in model_names if x.startswith("MiniCPM")]:
    r = bias_df.loc[m]
    gap = best_qwen_none - r["NONE recall before"]          # vs the best Qwen, same held-out half
    gained = r["NONE recall after"] - r["NONE recall before"]
    lost = r["tool pick before"] - r["tool pick after"]
    if not gap > 0.03:
        kind = "not a real gap: its NONE recall is within 3 points of the best Qwen"
    elif gained >= 0.5 * gap and lost <= gained / 3:
        kind = (f"mostly a fixable BIAS toward calling tools: one offset closes {gained / gap:.0%} of the "
                f"{gap:.0%} gap for {lost:.1%} of tool picking. LoRA should fix it easily")
    elif gained >= 0.2 * gap:
        kind = (f"PARTLY bias: one offset closes {gained / gap:.0%} of the {gap:.0%} gap, the rest is real "
                f"confusion. LoRA needs hard 'near-miss' negative examples")
    else:
        kind = (f"a REAL weakness, not a bias: one offset closes only {max(gained, 0) / gap:.0%} of the "
                f"{gap:.0%} gap. LoRA needs many 'no tool fits' examples")
    print(f"- {m}: the NONE gap is {kind}. "
          f"Test accuracy with the offset: {r['test acc before']:.1%} -> {r['test acc after']:.1%}.")

if diag_rows:
    dd = pd.DataFrame(diag_rows).dropna(subset=["accuracy"])
    for m in dd["model"].unique():
        g = dd[dd["model"] == m].set_index("format")
        base = g.loc["Answer: X"] if "Answer: X" in g.index else None
        best = g["accuracy"].idxmax()
        if base is None:
            continue
        delta = g.loc[best, "accuracy"] - base["accuracy"]
        if best != "Answer: X" and delta >= 0.08:   # 100 questions: smaller gaps are within noise
            print(f"- {m}: the answer format mattered: '{best}' scores {g.loc[best, 'accuracy']:.0%} vs "
                  f"{base['accuracy']:.0%} on the sample (letter mass {g.loc[best, 'letter mass']:.2f} vs "
                  f"{base['letter mass']:.2f}). Rerun the main comparison with that format before quoting numbers.")
        else:
            print(f"- {m}: low letter mass did NOT change its answers (best format within "
                  f"{abs(delta):.0%} of the main run), so the main-run accuracy stands.")

## 8. Kev's own benchmarks: the same questions Kev and Jev were scored on

Kev's evaluation suites are public and frozen (checksums in each manifest). This section scores our models on two of them with **Kev's own data files and Kev's own metric code** (`kev/metrics.py`), so the numbers line up with Kev's published ones:

- **transfer-v4 "New Sources"**: Kev's headline test. 656 clean questions from datasets Kev never trained on (MMLU, SciQ, QNLI, PAWS, Emotion, TweetEval-offensive, held-out policies and rules), development and test splits.
- **When2Call** (from `devtools-v1`): tool routing built on BFCL. Given the tools and a user message, the right next step is one of: call a tool, ask for missing information, say it can't help, or answer directly. 150 questions per split.

Our models answer each question exactly as in the main run (lettered options, one forward pass, probabilities read from the letter logits). Kev and Jev numbers are copied from Kev's README, model cards and committed reports at the pinned commit. Caveats: the public datasets in transfer-v4 (MMLU, SciQ, …) may have been in any model's pre-training, including Kev's Qwen bases; and Kev's published numbers use fp32 while ours run in fp16 on T4.

In [ ]:
import hashlib, sys

KEV_COMMIT = "1f04cfa28e6a708485b28a9585010b9e436b776e"
KEV_RAW = f"https://raw.githubusercontent.com/jaredpalmer/kev/{KEV_COMMIT}/"
KEV_DIR = os.path.join(OUT_DIR, "kev_src")
KEV_SPLITS = ["development", "test"]
KEV_FILES = ["kev/api.py", "kev/metrics.py",
             "evals/v4/transfer-v4/manifest.json", "evals/v4/transfer-v4/development.jsonl", "evals/v4/transfer-v4/test.jsonl",
             "evals/devtools-v1/manifest.json", "evals/devtools-v1/development.jsonl", "evals/devtools-v1/test.jsonl"]
for rel in KEV_FILES:
    dst = os.path.join(KEV_DIR, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(KEV_RAW + rel, dst)
open(os.path.join(KEV_DIR, "kev", "__init__.py"), "a").close()
if KEV_DIR not in sys.path:
    sys.path.insert(0, KEV_DIR)
from kev.api import render, option_text, question_keys      # how Kev turns state/options into text, and its answer keys
from kev.metrics import metrics as kev_metrics, grouped_metrics as kev_grouped   # Kev's scoring code


def load_kev_split(suite, split):
    path = os.path.join(KEV_DIR, suite, f"{split}.jsonl")
    manifest = json.load(open(os.path.join(KEV_DIR, suite, "manifest.json")))
    if hashlib.sha256(open(path, "rb").read()).hexdigest() != manifest["files"][f"{split}.jsonl"]["sha256"]:
        raise ValueError(f"checksum mismatch: {path}")
    return [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]


KEV_SUITES = {
    "transfer-v4": {s: load_kev_split("evals/v4/transfer-v4", s) for s in KEV_SPLITS},
    "When2Call":   {s: [r for r in load_kev_split("evals/devtools-v1", s) if r["_meta"]["source"] == "when2call"]
                    for s in KEV_SPLITS},
}
for suite, splits in KEV_SUITES.items():
    print(suite, {s: len(r) for s, r in splits.items()}, "records")

# Published numbers at KEV_COMMIT: (development, test); None = not published.
KEV_PUBLISHED = {
    "transfer-v4": {   # README "Accuracy: New Sources" and "Brier: New Sources"
        "Kev-0.8B": {"acc": (0.648, 0.697), "brier": (0.481, 0.416)},
        "Kev-4B":   {"acc": (0.817, 0.838), "brier": (0.269, 0.242)},
        "Kev-9B":   {"acc": (0.820, 0.852), "brier": (0.289, 0.217)},
        "Kev-27B":  {"acc": (0.851, 0.889), "brier": (0.225, 0.156)},
        "Jev":      {"acc": (0.857, None),  "brier": (0.211, None)},
    },
    "When2Call": {     # model cards and committed reports (runs/*/report.json, tasks.when2call_action)
        "Kev-0.8B": {"acc": (0.167, 0.133)},     # kev-0.8b card
        "Kev-4B":   {"acc": (0.660, None)},      # kev-4b card
        "Kev-9B":   {"acc": (None, 0.680)},      # runs/r27c-9b-cand-devtest (release candidate)
        "Jev":      {"acc": (0.760, None)},      # runs/jev-devtools-v1
    },
}

In [ ]:
KEV_SYSTEM = ("You are a decision model. Read the text, then answer the question by choosing exactly one of the "
              "lettered options. Reply with the option letter only.")


def kev_options(q):
    """[(answer key, text shown)] in display order. Yes/no questions show Yes first."""
    if q["type"] == "choice":
        return [(k, option_text(k, d)) for k, d in q["criteria"].items()]
    if q["type"] == "noul":
        crit = q.get("criteria") or {}
        return [("true", option_text("Yes", crit.get("true"))), ("false", option_text("No", crit.get("false")))]
    return [(str(i), render(c)) for i, c in enumerate(q["criteria"])]          # score: lowest -> highest


def kev_prompt(tok, record, q, opts):
    instructions = render(q.get("instructions")) or "Choose the best option."
    lines = [f"({LETTERS[i]}) {text}" for i, (_, text) in enumerate(opts)]
    if q["type"] == "score":
        lines.append("(The options are ordered from lowest to highest.)")
    user = (f"Text:\n{render(record['state'])}\n\nQuestion: {instructions}\n\nOptions:\n" + "\n".join(lines)
            + "\n\nReply with the letter of the best option only.")
    msgs = [{"role": "system", "content": KEV_SYSTEM}, {"role": "user", "content": user}]
    kwargs = dict(tokenize=False, add_generation_prompt=True, enable_thinking=False)
    try:
        text = tok.apply_chat_template(msgs, **kwargs)
    except Exception:
        text = tok.apply_chat_template([{"role": "user", "content": KEV_SYSTEM + "\n\n" + user}], **kwargs)
    return text + PREFILL


def kev_rows(record, probs_by_q):
    """One row per question, built the way kev.benchmark.prediction_rows does."""
    meta, rows = record["_meta"], []
    for qid, q in record["questions"].items():
        keys = question_keys(q["type"], q.get("criteria"))
        y = keys.index(q["label"]) if q["type"] == "choice" else int(q["label"])
        p = np.array([probs_by_q[qid][k] for k in keys], dtype=float)
        rows.append({"id": meta["id"], "question": qid, "source": meta["source"], "task": q["src"], "type": q["type"],
                     "variant": meta["variant"], "keys": keys, "label": y, "p": (p / p.sum()).tolist()})
    return rows


@torch.inference_mode()
def _kev_score(name, repo, dtype):
    tok, model = load_model(repo, dtype)
    dev = input_device(model)
    out = {}
    try:
        letter_ids = None
        for suite, splits in KEV_SUITES.items():
            for split, records in splits.items():
                rows = []
                for record in tqdm(records, desc=f"{name} {suite} {split}"):
                    probs_by_q = {}
                    for qid, q in record["questions"].items():
                        opts = kev_options(q)
                        if len(opts) > len(LETTERS):
                            raise ValueError(f"{record['_meta']['id']}: {len(opts)} options")
                        prompt = kev_prompt(tok, record, q, opts)
                        if letter_ids is None:
                            letter_ids = letter_token_ids(tok, prompt)[1]
                        ids = tok(prompt, add_special_tokens=False, return_tensors="pt")["input_ids"].to(dev)
                        lp = next_token_logprobs(model, ids)[letter_ids[:len(opts)]].double()
                        if not torch.isfinite(lp).all():
                            raise FloatingPointError(f"non-finite logits on {record['_meta']['id']}")
                        p = torch.softmax(lp, 0).cpu().numpy()
                        probs_by_q[qid] = {key: float(p[i]) for i, (key, _) in enumerate(opts)}
                    rows += kev_rows(record, probs_by_q)
                out[f"{suite}|{split}"] = rows
    finally:
        del model
        free_memory()
    return out


def kev_score_model(name, repo):
    for i, dtype in enumerate(DTYPES):                 # same fp16 -> fp32 fallback as the main run
        try:
            return _kev_score(name, repo, dtype)
        except FloatingPointError as e:
            if i == len(DTYPES) - 1:
                raise
            print(f"{name}: {e} in {dtype}; rerunning in {DTYPES[i + 1]}")


kev_results = {}
for name in model_names:
    path = os.path.join(OUT_DIR, f"kev_results_{name}.json")
    cached = [path] + sorted(glob.glob(f"/kaggle/input/**/kev_results_{name}.json", recursive=True))
    path = next((c for c in cached if os.path.exists(c)), path)
    if os.path.exists(path):
        kev_results[name] = json.load(open(path))
        print(f"{name}: loading cached Kev-suite results")
        continue
    try:
        kev_results[name] = kev_score_model(name, MODELS[name])
        with open(path, "w") as f:
            json.dump(kev_results[name], f)
    except Exception as e:
        print(f"!! {name} failed on the Kev suites: {type(e).__name__}: {e}")
        free_memory()
    if DELETE_WEIGHTS_AFTER:
        delete_cached_weights(MODELS[name])

In [ ]:
def fmt(dev, test):
    f = lambda v: "  -  " if v is None else f"{v:.3f}"
    return f"{f(dev)} / {f(test)}"


kev_tables = {}
for suite in KEV_SUITES:
    table = []
    for name in model_names:
        if name not in kev_results:
            continue
        m = {s: kev_metrics([r for r in kev_results[name][f"{suite}|{s}"] if r["variant"] == "clean"])
             for s in KEV_SPLITS}
        table.append({"system": name, "trained on it?": "no (zero-shot)",
                      "accuracy dev / test": fmt(m["development"]["acc"], m["test"]["acc"]),
                      "Brier dev / test": fmt(m["development"]["brier"], m["test"]["brier"]),
                      "_acc_test": m["test"]["acc"], "_acc_dev": m["development"]["acc"]})
    for system, vals in KEV_PUBLISHED[suite].items():
        acc, brier = vals["acc"], vals.get("brier", (None, None))
        table.append({"system": system, "trained on it?": "no (held out)" if suite == "transfer-v4" else "no (eval-only)",
                      "accuracy dev / test": fmt(*acc), "Brier dev / test": fmt(*brier),
                      "_acc_test": acc[1], "_acc_dev": acc[0]})
    df = pd.DataFrame(table).set_index("system")
    kev_tables[suite] = df
    df.drop(columns=["_acc_test", "_acc_dev"]).to_csv(os.path.join(OUT_DIR, f"kev_suite_{suite}.csv"))
    print(f"\n=== {suite} ({'656 clean questions per split' if suite == 'transfer-v4' else '150 questions per split'}) ===")
    display(df.drop(columns=["_acc_test", "_acc_dev"]))

# Per-source accuracy on transfer-v4 (test), ours only: where each model is strong or weak.
by_task = {}
for name in kev_results:
    rows = [r for r in kev_results[name]["transfer-v4|test"] if r["variant"] == "clean"]
    by_task[name] = {t: v["acc"] for t, v in kev_grouped(rows, "task").items()}
print("\ntransfer-v4 test accuracy by source (ours):")
display(pd.DataFrame(by_task))

# When2Call: which kind of next step does each model get right?
print("\nWhen2Call test: accuracy by correct answer (tool_call / request_for_info / cannot_answer):")
w2c = {}
for name in kev_results:
    rows = kev_results[name]["When2Call|test"]
    w2c[name] = {k: float(np.mean([int(np.argmax(r["p"]) == r["label"]) for r in rows if r["keys"][r["label"]] == k]))
                 for k in ("tool_call", "request_for_info", "cannot_answer")}
display(pd.DataFrame(w2c))

In [ ]:
# Plain-English read-out against Kev / Jev: each comparison uses the split the published number exists for
print("HOW OUR ZERO-SHOT MODELS COMPARE WITH KEV (fine-tuned) AND JEV\n")
for suite, df in kev_tables.items():
    print(f"{suite}:")
    refs = [s for s in df.index if s.startswith("Kev") or s == "Jev"]
    for name in [n for n in df.index if n in kev_results]:
        parts = []
        for ref in refs:
            for split, col in (("test", "_acc_test"), ("dev", "_acc_dev")):
                theirs = df.loc[ref, col]
                if theirs is not None and not pd.isna(theirs):
                    parts.append(f"{ref} {df.loc[name, col] - theirs:+.3f} ({split})")
                    break
        print(f"  {name:12s} " + "   ".join(parts))
    print()
print("Each number is our accuracy minus theirs, in points of accuracy (+0.050 = 5 points better).")

## 9. Native-format check: does MiniCPM5 decline better in its own tool-calling format?

Everything so far asked the models to pick a *letter*. Natively, a tool-calling model gets the tools through its chat template and either writes a tool call or answers in plain text. If MiniCPM5 declines and asks for missing information well in that native format, its weakness above is partly our framing; if it still calls a tool on everything, the weakness is real and the LoRA has to teach the "can this call actually work?" check.

The same questions are run both ways:
- **When2Call test** (150): should the assistant call a tool (50), ask for missing info (50), or say it can't help (50)?
- **BFCL** (60 + 60 from the main run): a tool fits, or no tool fits.

A native answer counts as a tool call if it contains MiniCPM5's `<function name=...>` or Qwen's `<tool_call>` / `<function=...>` markup (the formats SGLang's parsers use). Generation is greedy, thinking off.

In [ ]:
import re

RUN_NATIVE = False    # done in the first run (untrained models only); True re-runs it (~30-45 min)
NATIVE_MODELS = [m for m in ("MiniCPM5-1B", "MiniCPM5-2B", "Qwen3.5-4B") if m in MODELS] if RUN_NATIVE else []
NATIVE_BFCL_PER_GROUP = 60
NATIVE_MAX_NEW_TOKENS = 48
CALL_RE = re.compile(r"<tool_call>|<function[\s=]")
NAME_RES = [re.compile(r"<function\s+name=[\"']([^\"']+)"), re.compile(r"<function=([^>\s]+)"),
            re.compile(r'"name"\s*:\s*"([^"]+)"')]
TYPE_MAP = {"dict": "object", "float": "number", "tuple": "array", "any": "string"}   # BFCL type names -> JSON Schema


def to_schema(o):
    if isinstance(o, dict):
        return {k: (TYPE_MAP.get(v, v) if k == "type" and isinstance(v, str) else to_schema(v)) for k, v in o.items()}
    if isinstance(o, list):
        return [to_schema(x) for x in o]
    return o


def tool_spec(fn):
    params = to_schema(fn.get("parameters") or {"type": "object", "properties": {}})
    return {"type": "function", "function": {"name": fn["name"], "description": str(fn.get("description", "")),
                                             "parameters": params}}


def parse_tools(raw_list):
    specs = []
    for s in raw_list:
        try:
            fn = json.loads(s) if isinstance(s, str) else s
        except json.JSONDecodeError:            # a few When2Call records list no tools ("none")
            continue
        if isinstance(fn, dict) and "name" in fn:
            specs.append(tool_spec(fn))
    return specs


# --- the cases, identical for every model ---
native_cases = []
for r in KEV_SUITES["When2Call"]["test"]:
    q = next(iter(r["questions"].values()))
    native_cases.append({"set": "When2Call", "id": r["_meta"]["id"], "gold": q["label"],
                         "messages": [{"role": "user", "content": r["state"]["user_message"]}],
                         "tools": parse_tools(r["state"]["available_tools"])})

bfcl_raw = {x["id"]: x for cat in SAMPLES_PER_CATEGORY for x in fetch_jsonl(f"BFCL_v4_{cat}.json")}
nrng = random.Random(SEED + 2)
pool_rel = [e for e in examples if e["category"] == "live_multiple" and e["id"] in shared]
pool_irr = [e for e in examples if e["category"] in ("irrelevance", "live_irrelevance") and e["id"] in shared]
for e in nrng.sample(pool_rel, min(NATIVE_BFCL_PER_GROUP, len(pool_rel))) + \
         nrng.sample(pool_irr, min(NATIVE_BFCL_PER_GROUP, len(pool_irr))):
    raw = bfcl_raw[e["id"]]
    native_cases.append({"set": "BFCL", "id": e["id"], "gold": e["options"][e["gold_idx"]],
                         "messages": raw["question"][0], "tools": [tool_spec(f) for f in raw["function"]]})
print(pd.Series([c["set"] for c in native_cases]).value_counts().to_dict(), "cases")


@torch.inference_mode()
def native_answer(tok, model, case):
    kwargs = dict(tokenize=False, add_generation_prompt=True, enable_thinking=False)
    try:
        text = tok.apply_chat_template(case["messages"], tools=case["tools"] or None, **kwargs)
    except Exception:                            # template can't take this conversation/tool list
        text = tok.apply_chat_template([{"role": "user", "content": case["messages"][-1]["content"]}],
                                       tools=case["tools"] or None, **kwargs)
    ids = tok(text, add_special_tokens=False, return_tensors="pt")["input_ids"].to(input_device(model))
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    out = model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=NATIVE_MAX_NEW_TOKENS,
                         do_sample=False, pad_token_id=pad)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=False)


def classify(text):
    called = bool(CALL_RE.search(text))
    name = None
    if called:
        for rx in NAME_RES:
            m = rx.search(text)
            if m:
                name = m.group(1)
                break
    return called, name


native_results = {}
for name in NATIVE_MODELS:
    path = os.path.join(OUT_DIR, f"native_results_{name}.json")
    cached = [path] + sorted(glob.glob(f"/kaggle/input/**/native_results_{name}.json", recursive=True))
    path = next((c for c in cached if os.path.exists(c)), path)
    if os.path.exists(path):
        native_results[name] = json.load(open(path))
        print(f"{name}: loading cached native results")
        continue
    try:
        tok, model = load_model(MODELS[name], DTYPES[0])
        try:
            first = native_cases[0]
            print(f"--- {name}: prompt tail and first answer ---")
            print(tok.apply_chat_template(first["messages"], tools=first["tools"] or None, tokenize=False,
                                          add_generation_prompt=True, enable_thinking=False)[-400:])
            rows = []
            for case in tqdm(native_cases, desc=f"{name} native"):
                text = native_answer(tok, model, case)
                called, tool = classify(text)
                rows.append({"set": case["set"], "id": case["id"], "gold": case["gold"], "text": text,
                             "called": called, "tool": tool})
            print(rows[0]["text"][:300])
        finally:
            del model
            free_memory()
        native_results[name] = rows
        with open(path, "w") as f:
            json.dump(rows, f)
    except Exception as e:
        print(f"!! {name} native check failed: {type(e).__name__}: {e}")
        free_memory()
    if DELETE_WEIGHTS_AFTER:
        delete_cached_weights(MODELS[name])

In [ ]:
def mcq_when2call(name):
    """Letter-format answers from section 8 for the same When2Call test items: id -> predicted key."""
    rows = kev_results[name]["When2Call|test"]
    return {r["id"]: r["keys"][int(np.argmax(r["p"]))] for r in rows}


def rate(xs):
    return float(np.mean(xs)) if len(xs) else float("nan")


native_table, examples_text = {}, {}
for name, rows in native_results.items():
    w = [r for r in rows if r["set"] == "When2Call"]
    b = [r for r in rows if r["set"] == "BFCL"]
    mcq = mcq_when2call(name) if name in kev_results else {}
    res_m = res[res["model"] == name].set_index("id")
    for fmt in ("native", "letters"):
        if fmt == "native":
            w_call = {r["id"]: r["called"] for r in w}
            b_call = {r["id"]: r["called"] for r in b}
            b_tool_ok = {r["id"]: r["called"] and r["tool"] == r["gold"] for r in b if r["gold"] != "NONE"}
        else:
            if not mcq:
                continue
            w_call = {r["id"]: mcq[r["id"]] == "tool_call" for r in w if r["id"] in mcq}
            b_call, b_tool_ok = {}, {}
            for r in b:
                rr = res_m.loc[r["id"]]
                pred = rr["pred_idx"]
                b_call[r["id"]] = pred != rr["n_options"] - 1
                if r["gold"] != "NONE":
                    b_tool_ok[r["id"]] = bool(pred == rr["gold_idx"])
        native_table[(name, fmt)] = {
            "When2Call: calls when it should": rate([w_call[r["id"]] for r in w if r["gold"] == "tool_call" and r["id"] in w_call]),
            "When2Call: holds back when info is missing": rate([not w_call[r["id"]] for r in w if r["gold"] == "request_for_info" and r["id"] in w_call]),
            "When2Call: holds back when it can't help": rate([not w_call[r["id"]] for r in w if r["gold"] == "cannot_answer" and r["id"] in w_call]),
            "When2Call: call vs no-call accuracy": rate([w_call[r["id"]] == (r["gold"] == "tool_call") for r in w if r["id"] in w_call]),
            "BFCL: calls when a tool fits": rate([b_call[r["id"]] for r in b if r["gold"] != "NONE"]),
            "BFCL: right tool when a tool fits": rate(list(b_tool_ok.values())),
            "BFCL: holds back when nothing fits": rate([not b_call[r["id"]] for r in b if r["gold"] == "NONE"]),
        }
    examples_text[name] = [r["text"].strip()[:160].replace("\n", " ") for r in w
                           if r["gold"] == "request_for_info" and not r["called"]][:3]

native_df = pd.DataFrame(native_table)
native_df.to_csv(os.path.join(OUT_DIR, "native_vs_letters.csv"))
display(native_df)

for name, texts in examples_text.items():
    print(f"\n{name}: what it says natively when info is missing (first 3 non-call answers):")
    for t in texts:
        print("   ", t)

print("\nCONCLUSION")
for name in NATIVE_MODELS:
    if (name, "native") not in native_table or (name, "letters") not in native_table:
        continue
    n, l = native_table[(name, "native")], native_table[(name, "letters")]
    hold_n = np.nanmean([n["When2Call: holds back when info is missing"], n["When2Call: holds back when it can't help"],
                         n["BFCL: holds back when nothing fits"]])
    hold_l = np.nanmean([l["When2Call: holds back when info is missing"], l["When2Call: holds back when it can't help"],
                         l["BFCL: holds back when nothing fits"]])
    call_n, call_l = n["When2Call: calls when it should"], l["When2Call: calls when it should"]
    if hold_n - hold_l >= 0.15 and call_n >= call_l - 0.10:
        verdict = ("holds back much better in its native format: the weakness is largely our letter framing. "
                   "The LoRA mainly needs to teach the answer format.")
    elif hold_n - hold_l >= 0.05:
        verdict = "holds back somewhat better natively, but the gap is mostly real. The LoRA must teach the decision itself."
    else:
        verdict = ("holds back no better natively: the weakness is real. "
                   "The LoRA must teach the 'can this call actually work?' check with many negative examples.")
    print(f"- {name}: holds back {hold_n:.0%} natively vs {hold_l:.0%} with letters; "
          f"calls when it should {call_n:.0%} vs {call_l:.0%}. It {verdict}")

## 10. Scorecard: the LoRA router against everything

One table with every number that matters. Models scored in this run use this run's numbers; the others come from the first comparison run (same questions: the sampling is seeded) and from Kev's published results. Paired confidence intervals for the router against the base model are in the head-to-head table above.

In [ ]:
# Numbers from the first run (zero-shot, same 1,140 BFCL questions and the same Kev splits).
FIRST_RUN = {
    "MiniCPM5-1B":  {"bfcl_acc": 0.729, "tool_pick": 0.830, "none_recall": 0.617, "w2c_test": 0.420, "tv4_dev": 0.625, "tv4_test": 0.652},
    "MiniCPM5-2B":  {"bfcl_acc": 0.769, "tool_pick": 0.935, "none_recall": 0.585, "w2c_test": 0.373, "tv4_dev": 0.657, "tv4_test": 0.654},
    "Qwen3.5-0.8B": {"bfcl_acc": 0.732, "tool_pick": 0.668, "none_recall": 0.802, "w2c_test": 0.340, "tv4_dev": 0.582, "tv4_test": 0.627},
    "Qwen3.5-4B":   {"bfcl_acc": 0.845, "tool_pick": 0.960, "none_recall": 0.717, "w2c_test": 0.527, "tv4_dev": 0.712, "tv4_test": 0.745},
    "Qwen3.5-9B":   {"bfcl_acc": 0.887, "tool_pick": 0.935, "none_recall": 0.833, "w2c_test": 0.500, "tv4_dev": 0.787, "tv4_test": 0.802},
}
PUBLISHED = {   # Kev README / model cards / committed reports at KEV_COMMIT
    "Kev-0.8B": {"w2c_test": 0.133, "tv4_dev": 0.648, "tv4_test": 0.697},
    "Kev-4B":   {"w2c_dev": 0.660, "tv4_dev": 0.817, "tv4_test": 0.838},
    "Kev-9B":   {"w2c_test": 0.680, "tv4_dev": 0.820, "tv4_test": 0.852},
    "Jev":      {"w2c_dev": 0.760, "tv4_dev": 0.857},
}

card = {}
for m in model_names:
    row = {"bfcl_acc": summary.loc[m, "accuracy"], "tool_pick": summary.loc[m, "tool pick acc (relevant)"],
           "none_recall": summary.loc[m, "NONE recall (irrelevant)"], "source": "this run"}
    if m in kev_results:
        for split, key in (("development", "dev"), ("test", "test")):
            for suite, short in (("transfer-v4", "tv4"), ("When2Call", "w2c")):
                rows_ = [r for r in kev_results[m][f"{suite}|{split}"] if r["variant"] == "clean"]
                row[f"{short}_{key}"] = kev_metrics(rows_)["acc"]
    card[m] = row
for m, vals in FIRST_RUN.items():
    if m not in card:
        card[m] = {**vals, "source": "first run"}
for m, vals in PUBLISHED.items():
    card[m] = {**vals, "source": "published"}

cols = ["bfcl_acc", "tool_pick", "none_recall", "w2c_dev", "w2c_test", "tv4_dev", "tv4_test", "source"]
scorecard = pd.DataFrame(card).T.reindex(columns=cols)
scorecard.columns = ["BFCL overall", "BFCL tool pick", "BFCL NONE recall", "When2Call dev", "When2Call test",
                     "transfer-v4 dev", "transfer-v4 test", "numbers from"]
scorecard.to_csv(os.path.join(OUT_DIR, "scorecard.csv"))
display(scorecard)

if ROUTER in card:
    r = card[ROUTER]
    print(f"\nTHE ROUTER ({ROUTER}, 2.5B, LoRA)")
    def vs(metric, other, label):
        a, b = r.get(metric), card.get(other, {}).get(metric)
        if a is None or b is None or pd.isna(a) or pd.isna(b):
            return
        print(f"  {label:28s} {a:.3f} vs {other} {b:.3f}  ({a - b:+.3f})")
    for other in ("MiniCPM5-2B", "Qwen3.5-4B", "Qwen3.5-9B"):
        vs("bfcl_acc", other, "BFCL overall")
    for other in ("MiniCPM5-2B", "Kev-0.8B", "Kev-9B", "Qwen3.5-9B"):
        vs("w2c_test", other, "When2Call test")
    for other in ("Kev-4B", "Jev"):
        vs("w2c_dev", other, "When2Call dev")
    for other in ("MiniCPM5-2B", "Kev-0.8B", "Kev-4B", "Kev-9B"):
        vs("tv4_test", other, "transfer-v4 test")
    vs("tv4_dev", "Jev", "transfer-v4 dev")
    print("\nWhen2Call has 150 questions per split: differences under ~8 points are within noise.")

## Next steps
- **MiniCPM5-2B matches or beats Qwen3.5-4B/9B** → that's the headline: same routing quality at a fraction of the size and latency. Build the LoRA on MiniCPM5-2B (or 1B if it's close enough and you want phone-sized).
- **MiniCPM5 clearly ahead in its size class** → build the LoRA dataset (tool-routing examples from xLAM / Glaive / your own CardSense tools), train on the answer-letter token only, then rerun this notebook with the adapter.
- **Poor calibration (high ECE) but good accuracy** → fine: temperature scaling fixes most of it. To check whether RL caused it, uncomment `MiniCPM5-1B-SFT` and rerun (finished models load from cache).
- **Section 7 says the NONE gap is mostly bias** → the LoRA plan is cheap: MiniCPM5 already knows *which* tool; it just needs to learn to decline. Include plenty of "no tool fits" and near-miss examples.
- **Section 8 (Kev's own suites)**: the honest yardstick. If zero-shot MiniCPM5-2B is already near Kev-4B on transfer-v4, a LoRA has a real shot at beating it; When2Call shows whether "call / ask / decline" is where the work is.
- **Section 7 says another answer format scores better** → set `PREFILL` to it, delete the old `results_*.json` and rerun so every model is scored the same new way.